In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab06.ipynb")

<table style="width: 100%;" id="nb-header">
    <tr style="background-color: transparent;"><td>
        <img src="https://data-88e.github.io/assets/images/blue_text.png" width="250px" style="margin-left: 0;" />
    </td><td>
        <p style="text-align: right; font-size: 10pt;"><strong>Economic Models</strong>, Fall 2025<br>
            Dr. Eric Van Dusen <br>
        Alan Liang <br>
        Amal Bhatnagar <br>
        Sreeja Apparaju </p></td></tr>
</table>

# Lab 6: Modeling Income Inequality

In [ ]:
from datascience import *
import numpy as np
import matplotlib.pyplot as plt
from sympy import Symbol, integrate
from scipy.optimize import minimize
%matplotlib inline 

def find_k(lnx, lny):
    """Determines the best value for k in the simplified Rao and Tam model.
    
    :param lnx: log of the income percentiles (out of 1)
    :param lny: log of the income shares (out of 1)
    :returns: the best value for k.
    """
    def obj(k):
        return np.mean(np.abs(lny - k*lnx))
    res = minimize(obj, 2.5)
    return res.x[0]

## The Lorenz Curve
There are many mathematical models used to model the Lorenz curve. For this homework, we will work with the simplified Rao-Tam Lorenz curve<sup>1</sup>, as defined by:
$$L(x)=x^k,\quad \text{where} \; k>1$$

<!-- BEGIN QUESTION -->

**Question 1:** Show that the simplified Rao-Tam Lorenz curve model satisfies the conditions for a Lorenz curve. 


_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- END QUESTION -->

### Fitting the Lorenz Curve to Data
Now, let's try to fit this model to the existing US data. From figure 3 of the [2022 *Income in the United States* report](https://www.census.gov/library/publications/2023/demo/p60-279.html), we get:

In [ ]:
# Read in and clean the data
# You should understand this code and its implications!
us = Table().read_table("us_income_2022.csv")
us

For each row of the table, `income_share` shows the proportion of total income an `income_group` owns. For example, the richest 20% (5th quintile) earned 52.1% of the total national income.

**Question 2.1:**
Let's convert this table to points on the Lorenz curve. Recall that the Lorenz curve's y-axis is the cumulative income share: at the 20th percentile, the total income share is 3.0%, while at the 40th percentile, the total income share is 8.5% + 3.0% = 11.5%. 

*Hint: use `np.cumsum`.*

In [ ]:
income_group = ...
cum_income_share = ...

us_cumulative = Table().with_columns("income_percentile", income_group, "cum_income_share", cum_income_share)
us_cumulative

In [ ]:
grader.check("q2_1")

**Question 2.2:**
In addition, the US census bureau reported that the top 5 percent of income earners' income share was 23.5%. What is the corresponding $(x,y)$ point to this data point? 


In [ ]:
x_val = ...
y_val = ...

In [ ]:
grader.check("q2_2")

**Question 2.3:**
Add this pair of coordinates to our table, by appending `income_group` and `cum_income_share` with `x` and `y`.


In [ ]:
income_group_with_x = ...
cum_income_share_with_y = ...

us_cumulative_extended = Table().with_columns(
    "income_percentile", income_group_with_x, 
    "cum_income_share", cum_income_share_with_y)
us_cumulative_extended

In [ ]:
grader.check("q2_3")

**Question 2.4:**
To more easily fit our data, we will take the natural log of the curve to make it linear. Which one of the equations below equals the natural log of the curve?
$$
\begin{aligned}
&(a) &&\ln{y} = k + \ln{x} \\
&(b) &&\ln{x} = k + \ln{y} \\
&(c) &&\ln{y} = k \ln{x} \\
&(d) &&\ln{x} = k \ln{y}
\end{aligned}
$$

Assign the letter corresponding to your answer to `q2_4` below.

In [ ]:
q2_4 = ...

In [ ]:
grader.check("q2_4")

**Question 2.5:**
Transform the data and append it to the new table. Use the [`np.log`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.log.html) function.


In [ ]:
log_x = ...
log_y = ...
us_log = Table().with_columns("log_income_percentile", log_x, "log_cum_income_share", log_y)
us_log

In [ ]:
grader.check("q2_5")

<!-- BEGIN QUESTION -->

**Question 2.6:**
Create a scatter plot of this table. Add a line of best fit by setting the `fit_line` parameter to be `True`.


In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Question 2.7:**
Now it's time to find $k$. Although this function is linear, it does not have an intercept so that the traditional `np.polyfit` method you've seen in previous lectures will not work. Instead, use the custom `find_k` function provided, which will find the best value of $k$. `find_k` takes two arguments: the array $\ln x$ and the array $\ln y$.

In [ ]:
k_hat = ...
k_hat

In [ ]:
grader.check("q2_7")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Question 2.8** Suppose two countries, A and B, have the following income distributions for their bottom 50% of the population:

Country A: bottom 50% own 25% of total income.
    
Country B: bottom 50% own 30% of total income.

Describe how the Lorenz curve for Country B compares to Country A. Then choose which country has lower income inequality.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### Gini Coefficients

Now that we have found $\hat{k}$, let's try to determine its theoretical Gini coefficient. Recall that: 

$$
\begin{aligned}
\text{Gini} &= \frac{\text{Area between line of equality and Lorenz Curve}}{\text{Area under line of equality}} \\
&= \frac{\int_0^1 x \text{d}x - \int_0^1 L(x) \text{d}x}{\int_0^1 x \text{d}x} \\ 
&= 1 - 2\int_0^1 L(x) \text{d}x
\end{aligned}$$

**Question 3.1:**
Calculate the gini coefficient using the formula from the last line above. We will use SymPy to help us integrate the $\int_0^1 L(x) \text{d}x$. In order to integrate something using SymPy, we can use `integrate(f(x), (x, a, b))` where f(x) is the function we want to integrate and a and b are the starting and ending limits of the integral. You can find more information [here](https://docs.sympy.org/latest/modules/integrals/integrals.html).


In [ ]:
x = ...
area_under_lorenz = ...
gini = ...
gini

In [ ]:
grader.check("q3_1")

<!-- BEGIN QUESTION -->

**Question 3.2:**
The actual Gini coefficient in 2022 was reported to be 0.488. Is your result different than the reported value? What are some sources of potential error?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### Applying to Another Country

Now that you've calculuated the Gini coefficient for the US using the simplified Rao and Tam model, let's redo this for another country. The default country to use is Norway, which can be found in the `norway_income.csv` file. If you would like to instead analyze any other country, you're welcome to do that as well. For example, the income distribution data for European countries can be found [here](https://ec.europa.eu/eurostat/databrowser/view/ilc_di01/default/table). You do not need to read in a csv for this part; feel free to directly copy over the values by hand from the data source. Make sure to use the data for 2022 if possible (to compare to the US data for 2022) and **indicate in your explanation for question 4.2 which country you picked.**

In [ ]:
my_country = Table().read_table("norway_income_2022.csv")
my_country

<!-- BEGIN QUESTION -->

**Question 4.1:**
What is the Gini coefficient of your chosen country for its most recent year? Include a scatter plot of your logged data.

**Do not overwrite any variables from previous questions** (except `x`). If you overwrite anything, some tests may fail when your assignment is graded.


In [ ]:
income_group_my_country = ...
cum_income_share_my_country = ...

my_country_cumulative = Table().with_columns(
    "income_percentile", income_group_my_country, 
    "cum_income_share", cum_income_share_my_country
)

log_x_my_country = ...
log_y_my_country = ...

my_country_log = Table().with_columns(
    "log_income_percentile", log_x_my_country, 
    "log_cum_income_share", log_y_my_country
)

# Create scatter plot of log income percentile and log cumulative income share, with fit_line = True
...

k_hat_my_country = ...
 
x = ...
gini_my_country = ...
gini_my_country

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Question 4.2:**
How do your results compare with the actual Gini coefficient reported for the country, and does it suggest anything about our model?


_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Question 4.3**: Please fill out the form <a href="https://forms.gle/tHJA3qZFPx4C9iSn9">here</a> and assign the secret word to `q4_3`. 

In [ ]:
q4_3 = ...

In [ ]:
# BEGIN TESTS

In [ ]:
assert q4_3 == "norway"

In [ ]:
# END TESTS

---
**References:**
1. Rao & Tam curve https://www.tandfonline.com/doi/ref/10.1080/02664768700000032
2. Income and Poverty in the US https://www.census.gov/library/publications/2019/demo/p60-266.html

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)